# Post Data Analysis

This is the final stage of the project before transitioning to group tasks. Before continuing, make sure you have trained around `40` model combinations, covering all model types (`MyNN`, `NN`, `RNN`, `LSTM`, and `GRU`).
The provided code will generate three plots:

- __Training Loss Comparison__ – This plot shows the training loss for each model combination, helping you identify which combination achieved the lowest train loss.
- __Evaluation Metrics Comparison__ – This plot compares all metrics across all model combinations, allowing you to determine which model performed best overall.
- __Predictions vs. Target Values__ – This plot visualizes how the predicted values compare to the actual target values, helping you assess how well the model fits the data.

Now, skip the `Do Not Edit` cell, generate the plots, and complete the analysis tasks. To generate all three plots, you will need to update the variables appropriately based on the `# TODO` instructions.

In [ ]:
# ------------------- #
# --- Do Not Edit --- #
# ------------------- #

import os
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from collections import defaultdict

class ModelTrainingLossVisualizer:
    """Plots per-epoch training loss curves for every trained model combo.

    Reads the train_loss.json file each Trainer run wrote (see
    train_model.py) from a directory tree shaped
    <base_path>/<dataset>/<model>/<activation>/<optimizer>/epochs-<N>/,
    and produces one figure per (dataset, model) pair, overlaying a curve
    per activation/optimizer/epochs combo so you can compare training
    dynamics within that model type.
    """

    def __init__(self, base_path, datasets, models, activations, optimizers, epochs):
        """Store the sweep dimensions to search for saved results.

        Args:
            base_path (str): Root directory results were written under
                (matches the cwd train_model.py was run from).
            datasets (list[str]): Dataset names to look for, e.g. ["ideal"].
            models (list[str]): Model class names, e.g. ["NN", "LSTM"].
            activations (list[str]): Activation names, e.g. ["relu", "tanh"].
            optimizers (list[str]): Optimizer names, e.g. ["adam", "sgd"].
            epochs (list[int]): Epoch counts that were swept over.
        """
        self.base_path = base_path
        self.datasets = datasets
        self.models = models
        self.activations = activations
        self.optimizers = optimizers
        self.epochs_options = epochs

    def load_training_losses(self):
        """Walk every combo in the sweep and load whichever train_loss.json files exist.

        Combos with no matching file (e.g. never trained, or trained under
        a different base_path) are silently skipped rather than raising.

        Returns:
            dict[str, dict[str, list[tuple[str, list[float]]]]]:
                losses[dataset][model] -> list of (label, loss_series) pairs,
                where label is "<activation>-<optimizer>-<epochs>" and
                loss_series is the per-epoch loss values for that combo.
        """
        losses = defaultdict(lambda: defaultdict(list))

        for dataset in self.datasets:
            for model in self.models:
                for activation in self.activations:
                    for optimizer in self.optimizers:
                        for epoch in self.epochs_options:
                            folder = os.path.join(
                                self.base_path,
                                dataset,
                                model,
                                activation,
                                optimizer,
                                f"epochs-{epoch}"
                            )
                            path = os.path.join(folder, "train_loss.json")
                            if not os.path.isfile(path):
                                continue
                            try:
                                with open(path, "r") as f:
                                    data = json.load(f)
                                loss_series = [entry["loss"] for entry in data["train_loss"]]
                                label = f"{activation}-{optimizer}-{epoch}"
                                losses[dataset][model].append((label, loss_series))
                            except Exception as e:
                                print(f"Error reading {path}: {e}")
        return losses

    def plot_training_loss_by_model(self, losses):
        """Render one training-loss figure per (dataset, model) pair.

        Each figure overlays every activation/optimizer/epochs combo found
        for that model as a separate line, so you can see which combo
        converged fastest or to the lowest loss.

        Args:
            losses (dict): Output of load_training_losses().
        """
        colors = sns.color_palette("tab10")

        for dataset, model_data in losses.items():
            print(f"\nDataset: {dataset}")
            for model, loss_lists in model_data.items():
                if not loss_lists:
                    continue
                plt.figure(figsize=(10, 5), dpi=120)
                for i, (label, loss_series) in enumerate(loss_lists):
                    color = colors[i % len(colors)]
                    plt.plot(
                        range(1, len(loss_series) + 1),
                        loss_series,
                        label=label,
                        color=color,
                        marker='o',
                        markersize=4,
                        linewidth=1.5
                    )
                plt.title(f"{dataset.upper()} - {model} Training Loss", fontsize=14)
                plt.xlabel("Epoch", fontsize=12)
                plt.ylabel("Loss", fontsize=12)
                plt.xticks(fontsize=10)
                plt.yticks(fontsize=10)
                plt.legend(fontsize="small", frameon=True, framealpha=0.9, loc="best")
                plt.grid(True, linestyle="--", alpha=0.5)
                plt.tight_layout()
                plt.show()

class ModelEvaluationMetricsVisualizer:
    """Bar-charts MAE/RMSE/R² across every trained model combo, per dataset.

    Reads the evaluate_model.json file each Trainer.evaluate() call wrote
    (see train_model.py) from the same directory layout as
    ModelTrainingLossVisualizer, and produces one bar chart per
    (dataset, metric) pair with one bar per model/activation/optimizer/epochs
    combo -- useful for spotting the single best-performing combo overall,
    as opposed to comparing convergence behavior within one model type.
    """

    def __init__(self, base_path, datasets, models, activations, optimizers, epochs):
        """Store the sweep dimensions to search for saved results.

        Args:
            base_path (str): Root directory results were written under
                (matches the cwd train_model.py was run from).
            datasets (list[str]): Dataset names to look for, e.g. ["ideal"].
            models (list[str]): Model class names, e.g. ["NN", "LSTM"].
            activations (list[str]): Activation names, e.g. ["relu", "tanh"].
            optimizers (list[str]): Optimizer names, e.g. ["adam", "sgd"].
            epochs (list[int]): Epoch counts that were swept over.
        """
        self.base_path = base_path
        self.datasets = datasets
        self.models = models
        self.activations = activations
        self.optimizers = optimizers
        self.epochs_options = epochs

    def load_evaluation_metrics(self):
        """Walk every combo in the sweep and load whichever evaluate_model.json files exist.

        Combos with no matching file (e.g. never trained, or trained under
        a different base_path) are silently skipped rather than raising.

        Returns:
            dict[str, list[dict]]: metrics_data[dataset] -> list of dicts,
                one per combo found, each with keys "model", "label"
                ("<model>-<activation>-<optimizer>-<epochs>"), "mae",
                "rmse", and "r2".
        """
        metrics_data = defaultdict(list)

        for dataset in self.datasets:
            for model in self.models:
                for activation in self.activations:
                    for optimizer in self.optimizers:
                        for epoch in self.epochs_options:
                            folder = os.path.join(
                                self.base_path,
                                dataset,
                                model,
                                activation,
                                optimizer,
                                f"epochs-{epoch}"
                            )
                            path = os.path.join(folder, "evaluate_model.json")
                            if not os.path.isfile(path):
                                continue
                            try:
                                with open(path, "r") as f:
                                    data = json.load(f)
                                label = f"{model}-{activation}-{optimizer}-{epoch}"
                                metrics_data[dataset].append({
                                    "model": model,
                                    "label": label,
                                    "mae": data["mae"],
                                    "rmse": data["rmse"],
                                    "r2": data["r2"]
                                })
                            except Exception as e:
                                print(f"Error reading {path}: {e}")
        return metrics_data

    def plot_metrics_by_dataset(self, metrics_data):
        """Render one bar chart per (dataset, metric) pair, one bar per combo.

        Produces 3 figures per dataset (MAE, RMSE, R²), each with every
        found model/activation/optimizer/epochs combo as its own bar, so
        the best/worst combos for that metric are visible at a glance.

        Args:
            metrics_data (dict): Output of load_evaluation_metrics().
        """
        for dataset, items in metrics_data.items():
            if not items:
                continue

            print(f"\nDataset: {dataset}")
            sns.set(style="whitegrid")
            metric_names = ["mae", "rmse", "r2"]

            for metric in metric_names:
                labels = [item["label"] for item in items]
                values = [item[metric] for item in items]
                plt.figure(figsize=(14, 6), dpi=120)
                sns.barplot(x=labels, y=values, palette="Set2")
                plt.title(f"{dataset.upper()} - {metric.upper()} Across Configurations", fontsize=14)
                plt.xlabel("Model-Activation-Optimizer-Epoch", fontsize=12)
                plt.ylabel(metric.upper(), fontsize=12)
                plt.xticks(rotation=45, ha="right", fontsize=9)
                plt.yticks(fontsize=10)
                plt.tight_layout()
                plt.grid(True, axis="y", linestyle="--", alpha=0.5)
                plt.show()

class ForecastPlotter:
    """Plots a single trained combo's forecasts against ground truth for one building.

    Reads the predictions.json file a single Trainer.evaluate() call wrote
    (see train_model.py) for one specific dataset/model/activation/
    optimizer/epochs combo, and visualizes its forecasts for a chosen
    building and forecast index -- the finest-grained of the three
    visualizer classes here (the other two aggregate across *all* combos;
    this one drills into *one* combo's actual predictions).
    """

    def __init__(self, base_path, dataset, model, activation, optimizer, epochs):
        """Locate and load one combo's saved predictions.json file.

        Args:
            base_path (str): Root directory results were written under.
            dataset (str): Single dataset name, e.g. "ideal".
            model (str): Single model class name, e.g. "LSTM".
            activation (str): Single activation name, e.g. "relu".
            optimizer (str): Single optimizer name, e.g. "adam".
            epochs (int): Single epoch count this combo was trained for.
        """
        self.pred_path = os.path.join(
            base_path,
            dataset,
            model,
            activation,
            optimizer,
            f"epochs-{epochs}",
            "predictions.json"
        )
        self.data = self._load_predictions()

    def _load_predictions(self):
        """Load predictions.json for this combo, if it exists.

        Returns:
            dict or None: Parsed JSON keyed by building_id, or None if the
                file wasn't found (this combo hasn't been trained/evaluated
                yet, or base_path doesn't match where it was saved).
        """
        if not os.path.isfile(self.pred_path):
            print(f"File not found: {self.pred_path}")
            return None
        with open(self.pred_path, "r") as f:
            return json.load(f)

    def _set_plot_style(self):
        """Apply a shared seaborn style/font-size theme to the current figure."""
        plt.style.use("seaborn-v0_8-muted")
        plt.rcParams.update({
            "font.size": 13,
            "axes.labelsize": 14,
            "axes.titlesize": 15,
            "legend.fontsize": 12,
            "xtick.labelsize": 11,
            "ytick.labelsize": 11
        })

    def list_building_ids(self):
        """List every building_id available in this combo's loaded predictions.json.

        predictions.json only contains buildings from the test split (the
        last 20% of the dataset), so a building_id that exists in the raw
        dataset can still be missing here if it landed in the training split
        instead -- this is the fastest way to check what's actually usable
        for this combo before calling get_num_forecast_indices/plot_*.

        Returns:
        list[str]: Available building_ids, or [] if predictions weren't loaded.
        """
        if self.data is None:
            return {}
        counts = {building_id: len(entry["predictions"]) for building_id, entry in self.data.items()}
        for building_id, num_indices in counts.items():
            print(f"{building_id}: {num_indices} forecast indices (valid index: 0-{num_indices - 1})")
        return

    def get_num_forecast_indices(self, building_id):
        """Report how many forecast windows (indices) were saved for a building.

        Args:
            building_id (str): Building identifier as it appears as a key
                in predictions.json.

        Returns:
            int: Number of forecast indices available for building_id, or
                0 if predictions weren't loaded or building_id isn't present.
        """
        if self.data is None or building_id not in self.data:
            print(f"Missing data or building '{building_id}' not found.")
            return 0
        num = len(self.data[building_id]["predictions"])
        print(f"Building '{building_id}' has {num} forecast indices.")
        return num

    def plot_forecast_chain(self, building_id, index=0):
        """Plot the real load history immediately followed by the predicted forecast.

        Concatenates the real context window ("load") with the actual
        target values ("targets") into one continuous "Actual" line, then
        overlays the model's predicted values starting where the real
        history ends -- showing the forecast as a continuation of history,
        which makes it easy to see where the prediction diverges from
        what actually happened.

        Args:
            building_id (str): Building identifier as it appears as a key
                in predictions.json.
            index (int): Which saved forecast window to plot for this building.
        """
        if self.data is None or building_id not in self.data:
            print(f"Missing data or building '{building_id}' not found.")
            return

        self._set_plot_style()

        load = np.squeeze(np.array(self.data[building_id]["load"][index]))
        predictions = np.squeeze(np.array(self.data[building_id]["predictions"][index]))
        targets = np.squeeze(np.array(self.data[building_id]["targets"][index]))

        x_load = np.arange(len(load))
        x_future = np.arange(len(load), len(load) + len(predictions))
        full_target = np.concatenate([load, targets])
        x_target = np.arange(len(full_target))

        plt.figure(figsize=(14, 5))
        plt.plot(x_target, full_target, label="Actual (Load + Target)", linewidth=2.5, color="#4C72B0")
        plt.plot(x_future, predictions, label="Predicted", linewidth=2.5, color="#DD8452")
        plt.title(f"{building_id} | Forecast Chain | Index: {index}")
        plt.xlabel("Timestep")
        plt.ylabel("Energy Usage")
        plt.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
        plt.legend(frameon=False)
        plt.tight_layout(pad=2)
        plt.show()

    def plot_predicted_vs_target_only(self, building_id, index=0):
        """Plot predicted vs. actual values over just the forecast horizon.

        Unlike plot_forecast_chain, this omits the real context window
        entirely and overlays predictions directly on top of targets on
        the same x-axis (forecast step, not timestep) -- better for
        judging fit quality/shape since both lines share the same range.

        Args:
            building_id (str): Building identifier as it appears as a key
                in predictions.json.
            index (int): Which saved forecast window to plot for this building.
        """
        if self.data is None or building_id not in self.data:
            print(f"Missing data or building '{building_id}' not found.")
            return

        self._set_plot_style()

        predictions = np.squeeze(np.array(self.data[building_id]["predictions"][index]))
        targets = np.squeeze(np.array(self.data[building_id]["targets"][index]))

        if predictions.shape != targets.shape:
            print("Shape mismatch between predicted and target.")
            return

        x = np.arange(len(predictions))

        plt.figure(figsize=(12, 5))
        plt.plot(x, targets, label="Actual Target", linewidth=2.5, color="#4C72B0")
        plt.plot(x, predictions, label="Predicted", linewidth=2.5, color="#DD8452")
        plt.title(f"{building_id} | Predicted vs Target | Index: {index}")
        plt.xlabel("Forecast Step")
        plt.ylabel("Energy Usage")
        plt.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
        plt.legend(frameon=False)
        plt.tight_layout(pad=2)
        plt.show()

# ------------------- #
# --- Do Not Edit --- #
# ------------------- #

print("Finished!")

## Plot: Training Loss Comparison

Think about how best to use this plots for your final presentation. If you want to make any changes to the plots, you will have to change the class definition in the previous cell.

In [ ]:
# ------------------- #
# ------ Edit ------- #
# ------------------- #

if __name__ == "__main__":
    visualizer = ModelTrainingLossVisualizer(
        base_path=os.getcwd(), 
        datasets= , #TODO: Fill in the blanks
        models = , #TODO: Fill in the blanks
        activations = , #TODO: Fill in the blanks
        optimizers = , #TODO: Fill in the blanks
        epochs = , #TODO: Fill in the blanks
    )
    losses = visualizer.load_training_losses()
    visualizer.plot_training_loss_by_model(losses)
# ------------------- #
# ------ Edit ------- #
# ------------------- #

## Plot: Overall Metrics Comparison

In [ ]:
# ------------------- #
# ------ Edit ------- #
# ------------------- #

if __name__ == "__main__":
    visualizer = ModelEvaluationMetricsVisualizer(
        base_path=os.path.join(os.getcwd()), # TODO: 
        datasets= , #TODO: Fill in the blanks
        models = #TODO: Fill in the blanks,
        activations = #TODO: Fill in the blanks,
        optimizers = #TODO: Fill in the blanks,
        epochs = #TODO: Fill in the blanks
    )

    metrics = visualizer.load_evaluation_metrics()
    visualizer.plot_metrics_by_dataset(metrics)

# ------------------- #
# ------ Edit ------- #
# ------------------- #

## Plot: Predicted vs Target

Visualize the predictions for at least five distinct building IDs and compare how the predictions from each model align with the corresponding target values. For this plot, instead of providing variables in a list, you will need to provide individual values as strings, as shown in the example below.

```
dataset = "ideal" 
model = "NN"
activation = "relu"
optimizer = "adam"
epochs = 1
building_id = "home241"
index = 10
```

In [ ]:
# Use this method to figure out which building ids and indicies are available in the test set.

# ------------------- #
# ------ Edit ------- #
# ------------------- #

base_path = os.path.join(os.getcwd())
dataset = # TODO: 
model = # TODO:
activation = # TODO:
optimizer = # TODO:
epochs = # TODO:

plotter = ForecastPlotter(base_path, dataset, model, activation, optimizer, epochs)
plotter.list_building_ids()

# ------------------- #
# ------ Edit ------- #
# ------------------- #

In [ ]:
# ------------------- #
# ------ Edit ------- #
# ------------------- #

base_path = os.path.join(os.getcwd())
building_id = # TODO:
index = # TODO: 

plotter = ForecastPlotter(base_path, dataset, model, activation, optimizer, epochs)
plotter.plot_forecast_chain(building_id=building_id, index=index)
plotter.plot_predicted_vs_target_only(building_id=building_id, index=index)

# ------------------- #
# ------ Edit ------- #
# ------------------- #

## Task: Analysis

After generating all the plots, and based on the visualizations and metrics you have collected so far, please answer the following questions.

#### Best Performing Model

```
| Model  | Activation Function | Optimizer | Epochs | R² Score | MAE | MSE |
|--------|---------------------|-----------|--------|----------|-----|-----|
| NN     |                     |           |        |          |     |     |
| RNN    |                     |           |        |          |     |     |
| LSTM   |                     |           |        |          |     |     |
| GRU    |                     |           |        |          |     |     |
| MyNN   |                     |           |        |          |     |     |
```

1) Which model or models achieved the highest `R²` and the lowest `MAE` and `RMSE` scores?
2) How does your custom `MyNN` model compare to the other architectures in terms of evaluation metrics?
3) Which activation function and optimizer combination appears most frequently among the top-performing models?

```
### Training Time and Performance Metrics of Best Models

| Model  | Training Time (seconds)  | R² Score | MAE | MSE |
|--------|--------------------------|----------|-----|-----|
| NN     |                          |          |     |     |
| RNN    |                          |          |     |     |
| LSTM   |                          |          |     |     |
| GRU    |                          |          |     |     |
| MyNN   |                          |          |     |     |
```
1) Which model achieves the higest accuracy for least amount of time?
2) How does the training time of your custom `MyNN` model compare to the other architectures?

Considering the training time, evaluation metrics, and your own visualizations:
1) Which model would you consider the best overall?
2) Do you observe a clear trade-off between training time and evaluation metrics, where significant time savings can be achieved with only a minimal reduction in accuracy?

## Next Step: 

`/BuildingsBenchTutorial/Tutorials/Final-Project-Modules/Best-Overall-Model.md`